In [4]:
# read the csv file
import pandas as pd

# 步骤1: 读取原CSV文件（替换为实际文件名，如'your_file.csv'）
df = pd.read_csv('data_optim_add 1.csv')  # 或 pd.read_csv('your_file.csv')

# 步骤2: 筛选条件
filtered_df = df[(df['year'] == 2023) & 
                 (df['compensatedRelativePressure_mafiltered'].notna())]

# 步骤3: 创建新的DataFrame（已按csemCur分组，确保每个csemCur的不为空部分）
new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))

# # 步骤4: 可选保存到新CSV
# new_df.to_csv('new_filtered_df.csv', index=False)

# 查看结果
print("新DataFrame (new_df) 创建成功！")
print(f"形状: {new_df.shape}")
print(f"唯一csemCur数量: {new_df['csemCur'].nunique()}")
print(new_df.head())



新DataFrame (new_df) 创建成功！
形状: (625, 12)
唯一csemCur数量: 625
         csemCur  year  delta_pressure   cr  \
6   5.000201e+09  2023             NaN  0.0   
7   5.000252e+09  2023             NaN  0.0   
9   5.000252e+09  2023             NaN  0.0   
10  5.000252e+09  2023             NaN  0.0   
11  5.000252e+09  2023             NaN  0.0   

    compensatedRelativePressure_mafiltered  COPER  MANX     alpha  \
6                                 3.628186      0     0  0.004915   
7                                 3.551861      0     0  0.000000   
9                                 3.208579      0     0  0.181613   
10                                3.483369      0     0  0.022172   
11                                3.452095      0     0  0.018647   

    algo_critere_exploit_B_mafiltered  algo_critere_env_B_mafiltered  \
6                               False                          False   
7                               False                          False   
9                            

C:\Users\wu.wenbo\AppData\Local\Temp\ipykernel_26828\3819503712.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_df = filtered_df.groupby('csemCur', group_keys=False).apply(lambda x: x.dropna(subset=['compensatedRelativePressure_mafiltered']))


In [ ]:
# Model 1: no constraint
from gurobipy import Model, GRB, quicksum

# ---------------- Parameters ----------------
N = 3                          # CSEM number
T_years = 10                   # planning years
Y = 360                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

mu_COPER = 1.6
mu_MANT  = 4.2
V = [1, 1, 1]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
alpha_init = [0.005, 0.006, 0.0045]
L_total = 20
M = 0.1                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, ub=0.1, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 213 rows, 93 columns and 663 nonzeros
Model fingerprint: 0x0dd598c5
Variable types: 33 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+00]
  Objective range  [2e+04, 2e+07]
  Bounds range     [1e-01, 1e+00]
  RHS range        [4e-03, 1e+00]
Presolve removed 147 rows and 63 columns
Presolve time: 0.20s
Presolved: 66 rows, 30 columns, 221 nonzeros
Variable types: 10 continuous, 20 integer (20 binary)
Found heuristic solution: objective 1503176.5387

Root relaxation: objective 6.843682e+05, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 684368.248   

In [ ]:
# Model 2: budget constraint
from gurobipy import Model, GRB, quicksum

# ---------------- Parameters ----------------
N = 3                          # CSEM number
T_years = 10                   # planning years
Y = 360                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 2_000_000       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2
V = [1, 1, 1]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
alpha_init = [0.005, 0.006, 0.0045]
L_total = 20
M = 0.1                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, ub=0.1, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))


# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# 总成本预算约束
mdl.addConstr(total_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 214 rows, 93 columns and 753 nonzeros
Model fingerprint: 0x823b82c6
Variable types: 33 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+07]
  Objective range  [2e+04, 2e+07]
  Bounds range     [1e-01, 1e+00]
  RHS range        [4e-03, 2e+06]
Presolve removed 15 rows and 3 columns
Presolve time: 0.01s
Presolved: 199 rows, 90 columns, 750 nonzeros
Variable types: 30 continuous, 60 integer (60 binary)
Found heuristic solution: objective 1662329.3154

Root relaxation: objective 5.053920e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 505392.016    

In [ ]:
# Model 3: resources constraint

from gurobipy import Model, GRB, quicksum

# ---------------- Parameters ----------------
N = 3                          # CSEM number
T_years = 10                   # planning years
Y = 360                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 2_000_000       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2
V = [1, 1, 1]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
alpha_init = [0.005, 0.006, 0.0045]
L_total = 20
M = 0.1                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, ub=0.1, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))

# 4. 总任务数限制
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)

# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

Set parameter Username
Academic license - for non-commercial use only - expires 2026-11-05
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 214 rows, 93 columns and 723 nonzeros
Model fingerprint: 0x514c0572
Variable types: 33 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e-01, 1e+00]
  Objective range  [2e+04, 2e+07]
  Bounds range     [1e-01, 1e+00]
  RHS range        [4e-03, 2e+01]
Found heuristic solution: objective 1040190.8584
Presolve removed 15 rows and 3 columns
Presolve time: 0.02s
Presolved: 199 rows, 90 columns, 722 nonzeros
Variable types: 30 continuous, 60 integer (60 binary)

Root relaxation: objective 5.053920e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |

In [ ]:
# Model 4: budget and resources constraints
from gurobipy import Model, GRB, quicksum

# ---------------- Parameters ----------------
N = 3                          # CSEM number
T_years = 10                   # planning years
Y = 360                        # time units per year
P_high, P_low = 3.5, 3.2      # pressure thresholds
Delta_P = P_high - P_low       # = 0.3
C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

Total_budget = 2_000_000       # total budget
mu_COPER = 1.6
mu_MANT  = 4.2
V = [1, 1, 1]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
alpha_init = [0.005, 0.006, 0.0045]
L_total = 20
M = 0.1                        # Big-M (alpha range approximately 0~0.01)

mdl = Model("yearly_gas_maintenance")

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, lb=0, ub=0.1, name="alpha")  # alpha[i,t]: value for the year t

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(T_years):
        # No maintenance: alpha[t+1] = 1.01 * alpha[t]
        mdl.addConstr(alpha[i,t+1] >= 1.01 * alpha[i,t] - M * (x_COPER[i,t] + x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= 1.01 * alpha[i,t] + M * (x_COPER[i,t] + x_MANT[i,t]))
        
        # COPER maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_COPER
        alpha_coper = 1.01 * alpha[i,t] / mu_COPER
        mdl.addConstr(alpha[i,t+1] >= alpha_coper - M * (1 - x_COPER[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_coper + M * (1 - x_COPER[i,t]))
        
        # MANT maintenance: alpha[t+1] = 1.01 * alpha[t] / mu_MANT
        alpha_mant = 1.01 * alpha[i,t] / mu_MANT
        mdl.addConstr(alpha[i,t+1] >= alpha_mant - M * (1 - x_MANT[i,t]))
        mdl.addConstr(alpha[i,t+1] <= alpha_mant + M * (1 - x_MANT[i,t]))

# 4. 总任务数限制
mdl.addConstr(
    quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N) for t in range(T_years)) <= L_total,
    name="TotalTasksLimit"
)

# -------------- Objective: leak cost + maintenance cost --------------

# Annual leak cost: Leak_{i,t} = [Y * alpha[i,t] / Delta_P] * K[i] * C_leak
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)


# Maintenance cost
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)

total_cost = action_cost + leak_cost

# 总成本预算约束
mdl.addConstr(total_cost <= Total_budget, name="TotalBudgetLimit")

mdl.setObjective(total_cost, GRB.MINIMIZE)
mdl.Params.OutputFlag = 1  # Show solving process

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"Total cost: {mdl.objVal:.2f}")
    
    print("\n=== Maintenance Plan ===")
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  Year {t+1}: COPER (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            elif x_MANT[i,t].X > 0.5:
                print(f"  Year {t+1}: MANT (α from {alpha[i,t].X:.6f} to {alpha[i,t+1].X:.6f})")
            else:
                print(f"  Year {t+1}: No maintenance (α={alpha[i,t+1].X:.6f})")
else:
    print("No optimal solution found")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 215 rows, 93 columns and 813 nonzeros
Model fingerprint: 0xae9beb42
Variable types: 33 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+07]
  Objective range  [2e+04, 2e+07]
  Bounds range     [1e-01, 1e+00]
  RHS range        [4e-03, 2e+06]
Found heuristic solution: objective 1028413.7314
Presolve removed 15 rows and 3 columns
Presolve time: 0.02s
Presolved: 200 rows, 90 columns, 812 nonzeros
Variable types: 30 continuous, 60 integer (60 binary)

Root relaxation: objective 4.913744e+05, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 491374.401    